# Exercise 07 — Fine-tuning a Pre-trained Model

Last week you looked at what pretraining teaches a model: BERT fills in blanks and GPT-2 continues text. Neither of them can do a specific job out of the box, such as telling whether a film review is positive or negative. This week you **adapt** pre-trained models to tasks. There are three ways to do that, and this week has one notebook for each:

- **Fine-tuning** (this notebook): put a small task-specific layer on top of the model and keep training the *whole* model on labelled examples.
- **Prompting** (notebook 1): leave the model as it is and describe the task in its input.
- **Instruction tuning and preference alignment** (notebook 2): fine-tune a generative model on examples of good answers, and then on pairs of better and worse answers. This is how chat assistants are made.

In this notebook you fine-tune **DistilBERT** to classify the sentiment of **IMDB** movie reviews with the Hugging Face `Trainer`. You then compare it with a much cheaper alternative: a simple classifier trained on top of the *frozen* pre-trained model.

## What you will do
1. **The data**: IMDB reviews, balanced subsets, and how long the reviews are in tokens.
2. **The model**: DistilBERT with a classification head, and how well it does before any training.
3. **Fine-tuning**: tokenisation, dynamic padding and the `Trainer`.
4. **Frozen features**: logistic regression on DistilBERT's sentence vectors, compared with fine-tuning.
5. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **Heads up: this notebook downloads a model and a dataset**: DistilBERT (~270 MB) and IMDB (~85 MB). Both are cached after the first download. With the default settings, fine-tuning (Part 3) takes roughly 5–15 minutes on a laptop CPU and well under a minute on a GPU. Start it, then read ahead while it runs.

In [ ]:
%matplotlib inline
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

torch.manual_seed(0)

# Use the GPU if there is one; a CPU is enough for this notebook.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

### Settings

The values below keep the notebook fast on a CPU: we train on 2,000 of the 25,000 training reviews and cut every review off after 128 tokens. Leave them as they are for your first run.

In [ ]:
MODEL_NAME = "distilbert/distilbert-base-uncased"
N_TRAIN = 2000    # training reviews (out of 25,000)
N_TEST = 500      # test reviews (out of 25,000)
MAX_LENGTH = 128  # every review is truncated to this many tokens

# With a GPU (for example a free one on Colab or Kaggle, see COMPUTE_RESOURCES.md) you can afford much more.
# The line below takes a few minutes on a T4 and gives clearly better results:
# N_TRAIN, N_TEST, MAX_LENGTH = 10000, 2000, 256

## 1. The data: IMDB movie reviews

The [IMDB dataset](https://huggingface.co/datasets/stanfordnlp/imdb) (Maas et al., 2011) contains 50,000 movie reviews from imdb.com, each labelled **positive** (1) or **negative** (0). Half of them are for training and half for testing, and both halves are balanced: 12,500 positive and 12,500 negative reviews. (There is also an `unsupervised` split with 50,000 unlabelled reviews, which we do not use.)

The Hugging Face [`datasets`](https://huggingface.co/docs/datasets/index) library downloads it with one call. You get a `DatasetDict`: one `Dataset` per split, a table whose rows are examples and whose columns are *features*.

In [ ]:
imdb = load_dataset("stanfordnlp/imdb")
print(imdb)
print(imdb["train"].features)

In [ ]:
example = imdb["train"][0]  # one row is a dictionary: {"text": ..., "label": ...}
print("label:", example["label"])
print(example["text"][:600])

### Smaller, balanced subsets

Training on all 25,000 reviews would take hours on a CPU, so we work with `N_TRAIN` training and `N_TEST` test reviews. The obvious way to get them would be to take the first `N_TRAIN` rows, but first look at how the labels are ordered:

In [ ]:
print("labels of rows 12,495 to 12,504:", list(imdb["train"].select(range(12495, 12505))["label"]))

The dataset is **sorted by label**. The first 2,000 rows are all negative, and a model trained on them would only ever see one class.

**1.1 Complete `make_subset`**, which returns `n` *random* examples of a split.

Hints:
- `Dataset.shuffle(seed=...)` returns a shuffled copy of a dataset. Use the seed, so that everyone gets the same subsets.
- `Dataset.select(range(n))` keeps the first `n` rows.

In [ ]:
def make_subset(split, n, seed=0):
    # n random examples of a Dataset
    return ...  # TODO: shuffle the split with the seed, then keep the first n rows


train_data = make_subset(imdb["train"], N_TRAIN)
test_data = make_subset(imdb["test"], N_TEST)
print(train_data)

In [ ]:
# ✅ Check your subsets
assert len(train_data) == N_TRAIN and len(test_data) == N_TEST, "the subsets should have N_TRAIN and N_TEST examples"
for name, subset in [("train", train_data), ("test", test_data)]:
    share = np.mean(list(subset["label"]))
    print(f"{name}: {share:.0%} positive")
    assert 0.4 < share < 0.6, f"only {share:.0%} of the {name} reviews are positive. Did you shuffle?"
print("Looks good ✅")

### How long are the reviews?

A Transformer has a maximum input length (512 tokens for DistilBERT), and the cost of attention grows quadratically with the length. Before choosing `MAX_LENGTH`, look at how long the reviews are **in tokens**.

DistilBERT uses the same tokenizer as BERT last week: it lowercases the text and splits it into WordPiece subword tokens. `AutoTokenizer` loads the tokenizer that belongs to the model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

encoding = tokenizer("This movie was not good at all!")
print(encoding["input_ids"])
print(encoding.tokens())

**1.2 Compute the length in tokens of every training review**, without truncation.

Hint: the tokenizer also accepts a *list* of texts; `tokenizer(texts)["input_ids"]` is then a list with one list of ids per text. (You will see a warning that some sequences are longer than 512 tokens. You can ignore it here: we only count tokens and do not feed these sequences to the model.)

In [ ]:
train_texts = list(train_data["text"])
lengths = ...  # TODO: the number of tokens of every training review, as a NumPy array of shape (N_TRAIN,)

print(f"median {np.median(lengths):.0f} tokens, 90th percentile {np.percentile(lengths, 90):.0f}, longest {lengths.max()}")
print(f"{np.mean(lengths > MAX_LENGTH):.0%} of the reviews are longer than MAX_LENGTH = {MAX_LENGTH} tokens")

plt.figure(figsize=(8, 3))
plt.hist(lengths, bins=50)
plt.axvline(MAX_LENGTH, color="gray", linestyle="--", label="MAX_LENGTH")
plt.xlabel("tokens per review")
plt.ylabel("reviews")
plt.legend()
plt.show()

In [ ]:
# ✅ Check your lengths
assert isinstance(lengths, np.ndarray) and lengths.shape == (N_TRAIN,), "lengths should be a NumPy array with one entry per training review"
assert 150 < np.median(lengths) < 350, f"the median review is a bit over 200 tokens long, got {np.median(lengths):.0f}. Did you truncate?"
assert lengths.max() > 512, "some reviews are longer than 512 tokens. Did you truncate?"
print("Looks good ✅")

**1.3 With `MAX_LENGTH = 128` the model sees only the beginning of most reviews.** Why can sentiment classification still work reasonably well? What kind of review is likely to be misclassified because of the truncation, and how could you keep the *end* of each review instead?

---

*Your answer here:*

---

## 2. The model: DistilBERT with a classification head

[DistilBERT](https://arxiv.org/abs/1910.01108) (Sanh et al., 2019) is a smaller version of BERT: 6 instead of 12 Transformer encoder layers, trained (*distilled*) to imitate the outputs of the full BERT. According to its authors it keeps about 97% of BERT's performance on the GLUE benchmark while being 40% smaller and 60% faster, which makes it a good choice for a CPU.

`AutoModelForSequenceClassification` loads the pre-trained encoder and puts a **classification head** on top of it. For DistilBERT the head is a small MLP that is applied to the output vector of the first token, `[CLS]`:

```
[CLS] this movie was ... [SEP]
  │
  ▼  DistilBERT: 6 encoder layers (pre-trained)
h_[CLS]  (768 numbers)
  │
  ▼  pre_classifier: Linear(768, 768) + ReLU + dropout   ┐ new, randomly
  ▼  classifier:     Linear(768, 2)                      ┘ initialised
logits for (negative, positive)
```

The encoder weights come from pretraining. The head did not exist during pretraining, so it starts with **random** weights. The library warns you about this when you load the model (*"Some weights … were not initialized … You should probably TRAIN this model"*). That warning is expected here.

In [ ]:
id2label = {0: "negative", 1: "positive"}
label2id = {label: i for i, label in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
).to(device)
print(model.classifier, model.pre_classifier)

**2.1 How much of the model is new?** Count the parameters of the whole model and of the head (`model.pre_classifier` and `model.classifier`).

In [ ]:
n_total = ...  # TODO: number of parameters of the whole model
n_head = ...   # TODO: number of parameters of model.pre_classifier and model.classifier together

print(f"whole model: {n_total:,} parameters")
print(f"head:        {n_head:,} parameters ({n_head / n_total:.1%} of the model)")

In [ ]:
# ✅ Check your counts
assert n_head == (768 * 768 + 768) + (768 * 2 + 2), "the head has a 768 x 768 and a 768 x 2 linear layer, each with a bias"
assert 66_000_000 < n_total < 68_000_000, f"DistilBERT with the head has about 67 M parameters, got {n_total:,}"
print("Looks good ✅")

### How good is the model before training?

To measure that, you need two functions: one that makes predictions for a list of texts, and one that compares them with the true labels.

**2.2 Complete `predict`.** It runs the model on the texts in batches and returns the predicted label and the probability of *positive* for every text.

Hints:
- Tokenize a batch with `tokenizer(batch, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt")`. `padding=True` pads every text to the longest one *in the batch*, so the batch fits into one tensor.
- Move the inputs to the model's device with `.to(model.device)`, and call the model with `model(**inputs)`. The output has an attribute `.logits` of shape `(batch_size, 2)`.

In [ ]:
@torch.no_grad()
def predict(model, texts, batch_size=32):
    # The predicted label (0 or 1) and the probability of "positive" for every text, as two NumPy arrays
    model.eval()
    preds, probs = [], []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        inputs = ...  # TODO: tokenize the batch (see the hints) and move it to model.device
        logits = ...  # TODO: forward pass, (batch_size, 2)
        p = ...       # TODO: softmax over the two classes, (batch_size, 2)
        preds.append(p.argmax(dim=-1).cpu())
        probs.append(p[:, 1].cpu())
    return torch.cat(preds).numpy(), torch.cat(probs).numpy()

**2.3 Complete `compute_scores`.** Besides the accuracy and the F1 score, it reports the accuracy on the positive and on the negative reviews separately. On a balanced test set, these two numbers tell you whether the model is biased towards one class.

Hint: `sklearn.metrics` has `accuracy_score(y_true, y_pred)` and `f1_score(y_true, y_pred)` (F1 of the positive class). For the per-class accuracies, select the examples with a boolean mask such as `labels == 1`.

In [ ]:
def compute_scores(labels, preds):
    # Accuracy, F1 and the accuracy on the positive (label 1) and negative (label 0) examples
    labels, preds = np.asarray(labels), np.asarray(preds)
    return {
        "accuracy": ...,           # TODO
        "f1": ...,                 # TODO
        "accuracy_positive": ...,  # TODO: accuracy on the examples whose true label is 1
        "accuracy_negative": ...,  # TODO: accuracy on the examples whose true label is 0
    }

In [ ]:
# ✅ Check compute_scores on a small example
scores = compute_scores([1, 1, 1, 0, 0], [1, 1, 0, 0, 1])
assert np.isclose(scores["accuracy"], 3 / 5), "3 of the 5 predictions are correct"
assert np.isclose(scores["f1"], 2 / 3), "precision and recall are both 2/3, so F1 is 2/3"
assert np.isclose(scores["accuracy_positive"], 2 / 3), "2 of the 3 positive examples are correct"
assert np.isclose(scores["accuracy_negative"], 1 / 2), "1 of the 2 negative examples is correct"
print("Looks good ✅")

Now evaluate the model on the test set **before** any training. On a CPU this takes about a minute.

In [ ]:
test_texts = list(test_data["text"])
test_labels = np.array(list(test_data["label"]))


def show_scores(scores):
    print("  ".join(f"{name}: {value:.3f}" for name, value in scores.items()))


start = time.time()
preds_before, _ = predict(model, test_texts)
scores_before = compute_scores(test_labels, preds_before)
print(f"({time.time() - start:.0f} s)")
show_scores(scores_before)

In [ ]:
# ✅ Check your predict
assert preds_before.shape == (N_TEST,), "predict should return one prediction per text"
assert set(preds_before.tolist()) <= {0, 1}, "the predictions should be 0 or 1"
_, probs = predict(model, test_texts[:4])
assert probs.shape == (4,) and np.all((probs >= 0) & (probs <= 1)), "the probabilities should be between 0 and 1"
print("Looks good ✅")

**2.4 Look at the scores before training.** Why is the accuracy close to 50%, and why are the accuracies on the two classes so different? Pretraining has taught DistilBERT a lot about English. Why does none of that knowledge show up here?

---

*Your answer here:*

---

## 3. Fine-tuning with the `Trainer`

**Fine-tuning** means training the pre-trained model further, now on the task's labelled examples, with the usual cross-entropy loss. *All* weights are updated, the pre-trained ones as well as the new head. Because the pre-trained weights are already a very good starting point, a small learning rate and a few thousand examples are enough.

You could write the training loop yourself, as in weeks 4 to 6. The Hugging Face `Trainer` does it for you: batching, the optimiser and learning-rate schedule, moving data to the GPU, logging, evaluation and saving checkpoints. It needs the data in tokenized form.

### Tokenizing the dataset

`Dataset.map(function, batched=True)` applies a function to the dataset in batches and adds the columns that the function returns (here `input_ids` and `attention_mask`). The result is cached on disk, so running the cell again is fast.

**3.1 Complete `tokenize`.** Truncate to `MAX_LENGTH`, but do **not** pad: padding is done later, batch by batch.

In [ ]:
def tokenize(batch):
    # batch["text"] is a list of reviews
    return ...  # TODO: tokenize the texts, truncated to MAX_LENGTH, without padding


train_tok = train_data.map(tokenize, batched=True)
test_tok = test_data.map(tokenize, batched=True)
print(train_tok)

In [ ]:
# ✅ Check your tokenization
assert "input_ids" in train_tok.column_names and "attention_mask" in train_tok.column_names, "map should add input_ids and attention_mask"
sample_lengths = [len(ids) for ids in train_tok["input_ids"][:100]]
assert max(sample_lengths) <= MAX_LENGTH, "the sequences should be truncated to MAX_LENGTH"
assert min(sample_lengths) < MAX_LENGTH, "the sequences should not be padded yet"
print("Looks good ✅")

### Dynamic padding

To put sequences of different lengths into one tensor, they have to be padded. A **data collator** is the function that turns a list of examples into a batch. `DataCollatorWithPadding` pads every batch to the length of its *longest* sequence, and sets the `attention_mask` to 0 at the padding positions so that attention ignores them. It also renames `label` to `labels`, the name the model expects for the targets.

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

examples = [{key: train_tok[i][key] for key in ["input_ids", "attention_mask", "label"]} for i in range(4)]
print("lengths of the examples:", [len(example["input_ids"]) for example in examples])
batch = data_collator(examples)
print({name: tuple(tensor.shape) for name, tensor in batch.items()})

**3.2 Why pad each batch separately** to its longest sequence instead of padding every review to `MAX_LENGTH` once, when tokenizing? Would the result of the model be different?

---

*Your answer here:*

---

### Training arguments

`TrainingArguments` collects all settings of the training run. You do not need to change them now. Read through them, since you will set up runs like this one yourself in the project.

In [ ]:
training_args = TrainingArguments(
    output_dir="imdb-distilbert",     # logs (and checkpoints, if any) are written here
    num_train_epochs=1,               # one pass over the N_TRAIN training reviews
    per_device_train_batch_size=16,   # reviews per training step
    per_device_eval_batch_size=64,    # reviews per evaluation step (no gradients, so this can be larger)
    learning_rate=2e-5,               # much smaller than the 1e-3 of last week's training from scratch
    warmup_steps=0.1,                 # increase the learning rate from 0 over the first 10% of the steps ...
    lr_scheduler_type="linear",       # ... then decrease it linearly to 0
    weight_decay=0.01,                # AdamW's weight decay, a mild regularisation
    logging_steps=10,                 # record the training loss every 10 steps
    save_strategy="no",               # do not save checkpoints (each would be ~270 MB)
    report_to="none",                 # no external logging service (such as wandb or tensorboard)
    seed=0,
)

### The `Trainer`

During evaluation the `Trainer` collects the model's logits for the whole evaluation set and passes them to a `compute_metrics` function, as an object with two attributes: `eval_pred.predictions`, the logits as a NumPy array of shape `(N, 2)`, and `eval_pred.label_ids`, the labels of shape `(N,)`.

**3.3 Complete `compute_metrics`**, reusing your `compute_scores`, and **create the `Trainer`**.

The `Trainer` needs:
- `model`, `args` (the training arguments),
- `train_dataset` and `eval_dataset`: the *tokenized* datasets,
- `processing_class`: the tokenizer (it is saved together with the model),
- `data_collator` and `compute_metrics`.

In [ ]:
def compute_metrics(eval_pred):
    # Turn the logits into predicted labels and score them
    preds = ...  # TODO: the predicted label of every example, (N,)
    return compute_scores(eval_pred.label_ids, preds)


trainer = ...  # TODO: create the Trainer

**3.4 Fine-tune the model.** With the default settings this is 125 steps. It takes roughly 5–15 minutes on a laptop CPU and under a minute on a GPU, and a progress bar shows how far it is.

In [ ]:
start = time.time()
trainer.train()
print(f"Training took {time.time() - start:.0f} s")

history = [(log["step"], log["loss"]) for log in trainer.state.log_history if "loss" in log]
steps, losses = zip(*history)
plt.figure(figsize=(8, 3))
plt.plot(steps, losses)
plt.xlabel("training step")
plt.ylabel("training loss")
plt.show()

In [ ]:
scores_after = trainer.evaluate()
print(scores_after)

In [ ]:
# ✅ Check the result
assert scores_after["eval_accuracy"] > 0.75, f"the fine-tuned model should reach an accuracy above 75%, got {scores_after['eval_accuracy']:.1%}"
print(f"accuracy before fine-tuning: {scores_before['accuracy']:.1%}, after: {scores_after['eval_accuracy']:.1%}")
print("Looks good ✅")

**3.5 Compare the scores before and after fine-tuning.** The model saw only 2,000 labelled reviews, once each. Why is that enough? And why do we use a learning rate of `2e-5`, fifty times smaller than the `1e-3` you used to train the mini GPT last week? What could go wrong with a large learning rate?

---

*Your answer here:*

---

### Where does the model go wrong?

**3.6 Run the cell below.** It shows the test reviews that the fine-tuned model got wrong with the highest confidence.

In [ ]:
preds_after, probs_after = predict(model, test_texts)
confidence = np.where(preds_after == 1, probs_after, 1 - probs_after)
wrong = np.flatnonzero(preds_after != test_labels)
print(f"{len(wrong)} of {N_TEST} test reviews are misclassified\n")

for i in wrong[np.argsort(-confidence[wrong])][:3]:
    print(f"true: {id2label[test_labels[i]]}, predicted: {id2label[preds_after[i]]} with probability {confidence[i]:.2f}")
    print(test_texts[i][:1200])
    print()

**3.7 Why do you think the model got these reviews wrong?** Look for patterns, for example irony, mixed opinions, a verdict at the end, or labels you disagree with.

---

*Your answer here:*

---

## 4. Frozen features: a cheaper alternative

Fine-tuning updated all 67 M parameters. That needs a GPU for larger datasets, and if you have ten tasks, you end up with ten copies of the model. A much cheaper alternative is to keep the pre-trained model **frozen** and use it only as a **feature extractor**:

1. Run every review through the *pre-trained* DistilBERT once, without gradients.
2. Take the output vector of the `[CLS]` token as a fixed 768-dimensional representation of the review.
3. Train a simple classifier, here **logistic regression**, on these vectors.

This is how you used static word embeddings in week 3, but now with a contextual vector for the whole review.

The `model` above has been fine-tuned, so we load a fresh copy of the pre-trained encoder. `AutoModel` loads it *without* any head; its output has an attribute `.last_hidden_state` of shape `(batch_size, seq_len, 768)`.

In [ ]:
encoder = AutoModel.from_pretrained(MODEL_NAME).to(device)
encoder.eval()
print(f"{sum(p.numel() for p in encoder.parameters()):,} parameters, none of which will be trained")

**4.1 Complete `embed`**, which returns the `[CLS]` vector of every text.

Hint: tokenize exactly as in `predict`. The `[CLS]` token is at position 0 of every sequence.

In [ ]:
@torch.no_grad()
def embed(texts, batch_size=32):
    # The output vector of the [CLS] token for every text, as a NumPy array of shape (len(texts), 768)
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        inputs = ...  # TODO: tokenize the batch and move it to encoder.device
        hidden = ...  # TODO: the last hidden states of the encoder, (batch_size, seq_len, 768)
        vectors.append(...)  # TODO: the vectors at position 0, moved to the CPU
    return torch.cat(vectors).numpy()


start = time.time()
train_vectors = embed(train_texts)
test_vectors = embed(test_texts)
train_labels = np.array(list(train_data["label"]))
print(f"({time.time() - start:.0f} s)  train: {train_vectors.shape}, test: {test_vectors.shape}")

In [ ]:
# ✅ Check your embed
assert train_vectors.shape == (N_TRAIN, 768) and test_vectors.shape == (N_TEST, 768), "expected one 768-dimensional vector per review"
assert np.allclose(embed(test_texts[:2]), test_vectors[:2], atol=1e-4), "embedding the same text twice should give the same vector"
print("Looks good ✅")

**4.2 Train a logistic regression classifier** on the training vectors and evaluate it on the test vectors.

Hint: `LogisticRegression(max_iter=2000).fit(X, y)` trains it, and `.predict(X)` predicts labels. (`max_iter` is raised from its default of 100, which is not always enough for 768 features to converge.)

In [ ]:
classifier = ...    # TODO: a LogisticRegression fitted on train_vectors and train_labels
preds_frozen = ...  # TODO: its predictions for test_vectors

scores_frozen = compute_scores(test_labels, preds_frozen)
show_scores(scores_frozen)

In [ ]:
# ✅ Check your classifier
assert preds_frozen.shape == (N_TEST,), "expected one prediction per test review"
assert scores_frozen["accuracy"] > 0.65, f"logistic regression on the [CLS] vectors should be clearly better than chance, got {scores_frozen['accuracy']:.1%}"
print("Looks good ✅")

Here is everything side by side:

In [ ]:
n_logreg = classifier.coef_.size + classifier.intercept_.size
print(f"{'':<34}{'trained parameters':>20}{'accuracy':>10}")
for name, n_trained, accuracy in [
    ("pre-trained, random head", 0, scores_before["accuracy"]),
    ("frozen + logistic regression", n_logreg, scores_frozen["accuracy"]),
    ("fine-tuned", n_total, scores_after["eval_accuracy"]),
]:
    print(f"{name:<34}{n_trained:>20,}{accuracy:>10.1%}")

**4.3 Compare the two approaches.** Why does fine-tuning do better than logistic regression on frozen vectors? Name two situations in which you would still choose frozen features.

---

*Your answer here:*

---

## 5. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 5.1. What is updated during fine-tuning?

In full fine-tuning of a pre-trained model with a new classification head (as in Part 3), which weights are updated?

A. Only the classification head<br>
B. Only the last encoder layer and the head<br>
C. All weights of the encoder and the head<br>
D. Only the embedding layer<br>

**Answer:** 

---

### 5.2. Learning rate

Why are pre-trained Transformers usually fine-tuned with a small learning rate (around $10^{-5}$ to $5 \cdot 10^{-5}$)?

A. Large learning rates are not supported by AdamW<br>
B. To avoid destroying the knowledge stored in the pre-trained weights (catastrophic forgetting)<br>
C. Because the classification head is already trained<br>
D. Small learning rates make every training step faster<br>

**Answer:** 

---

### 5.3. Dynamic padding

What does `DataCollatorWithPadding` do?

A. It pads every sequence to the model's maximum length (512 tokens)<br>
B. It pads the sequences in each batch to the length of the longest sequence in that batch<br>
C. It removes padding by truncating every sequence to the shortest one<br>
D. It pads every sequence to the length of the longest sequence in the whole dataset<br>

**Answer:** 

---

### 5.4. Frozen features vs. fine-tuning

Which statement about frozen features (a classifier on top of a frozen pre-trained model) and fine-tuning is correct?

A. Frozen features update the encoder's weights, but only for one epoch<br>
B. Fine-tuning is usually more accurate, but needs a full copy of the model for every task<br>
C. Frozen features need more labelled data than fine-tuning<br>
D. Fine-tuning can only be applied to decoder-only models<br>

**Answer:** 

---

### 5.5. The classification head

Before fine-tuning, the DistilBERT classifier in Part 2 had an accuracy of about 50%. Why?

A. DistilBERT was pretrained on too little data<br>
B. The tokenizer truncated the reviews<br>
C. The classification head was newly added and randomly initialised<br>
D. The test set is unbalanced<br>

**Answer:** 